## Columns to Drop
| Column | Reason |
|--------|--------|
| `id` | Just a row index, no signal |
| `recorded_by` | Only 1 unique value across all rows |
| `num_private` | 98.7% zeros, near-zero correlation with everything |
| `scheme_name` | 48.5% missing + 2,695 unique values |
| `wpt_name` | 37,399 unique values, unlearnable |
| `subvillage` | 19,287 unique values, too granular |
| `ward` | 2,092 unique values, too granular |
| `date_recorded` | Replaced by `year_recorded` and `month_recorded` |
| `region_code` | Redundant with `region` |
| `district_code` | Redundant with `region_code` |
| `quantity_group` | Identical to `quantity` |
| `payment_type` | Identical to `payment` |
| `quality_group` | Identical to `water_quality` |
| `source_type` | Redundant with `source_class` |
| `waterpoint_type_group` | Redundant with `waterpoint_type` |
| `extraction_type_group` | Redundant with `extraction_type_class` |
| `management_group` | Redundant with `management` |
| `public_meeting` | Low predictive value |
| `scheme_management` | Very similar distribution to `management` |

---

## Missing Values to Fix
| Column | Issue | Fix |
|--------|-------|-----|
| `amount_tsh` | 70.1% zeros = missing | Replace 0 → NaN → fill with median |
| `construction_year` | 34.9% zeros = missing | Replace 0 → NaN → fill with median (1986) |
| `population` | 36% zeros = missing | Replace 0 → NaN → fill with median |
| `gps_height` | 34.4% zeros + negative values (-90) | Replace 0 and negatives → NaN → fill with median |
| `longitude` | 3.1% zeros = geographically impossible | Replace 0 → NaN → fill with region median |
| `funder` | 6.1% NaN | Fill with "Unknown" |
| `installer` | 6.1% NaN | Fill with "Unknown" |
| `permit` | 5.1% NaN | Fill with mode |

---

## Skewness & Outliers
| Column | Skewness | Kurtosis | Fix |
|--------|----------|----------|-----|
| `num_private` | 91.9 | 11137.3 | Drop entirely |
| `amount_tsh` | 57.8 | 4903.5 | log1p transform |
| `population` | 12.7 | 402.3 | log1p transform |

> Don't remove outliers — log1p transformation is enough to compress the scale.

---

## Feature Engineering
| New Column | Formula | Why |
|------------|---------|-----|
| `pump_age` | `year_recorded - construction_year` | Older pumps more likely to fail |
| `is_year_missing` | `1 if construction_year == 0` | Flag where year was missing |
| `is_pop_missing` | `1 if population == 0` | Flag where population was missing |
| `is_coord_missing` | `1 if longitude == 0` | Flag where coordinates were invalid |

---

## Strongest Predictors to Keep
| Feature | Why |
|---------|-----|
| `quantity` | Dry pumps are 96.9% non-functional — strongest predictor |
| `payment` | Annual payment → 75.2% functional, never pay → 44.9% functional |
| `water_quality` | Unknown quality → 84.1% non-functional |
| `waterpoint_type` | Dam → 85.7% functional, other → 13.2% functional |
| `extraction_type_class` | Other type → 80.8% non-functional |
| `management` | Private operator → 74.9%, unknown → 39.9% functional |
| `region` | Ranges from 78.2% to 29.8% functional across regions |
| `basin` | Lake Nyasa 65.4% vs Ruvuma 37.2% functional |
| `source_class` | Only 3 categories, clear signal |
| `gps_height` | Moderate geographic signal |
| `construction_year` | Strongest numerical predictor, curves clearly separated |
| `pump_age` | Engineered — directly measures operational age |

---

## Class Imbalance
- functional: 54.3% — non functional: 38.4% — needs repair: 7.3%
- Imbalance ratio: 7.47x
- **Fix:** use `class_weight='balanced'` so the model doesn't ignore the minority class